In [ ]:
import torch  
import torch.nn as nn  
import torch.nn.functional as F 
import math  
from typing import List, Tuple, Optional  


class DETR3D(nn.Module):
    """
    DETR3D: 基于3D到2D查询的多视角图像3D目标检测
    主要功能:
    - 从多视角图像中检测3D物体
    - 使用可学习的查询(query)机制
    - 通过Transformer结构进行特征交互
    """
    
    def __init__(self,
                 num_classes: int = 10,  # 检测类别数量(如汽车、行人等)
                 num_queries: int = 900,  # 查询数量(即最大检测目标数)
                 num_layers: int = 6,  # Transformer层数
                 hidden_dim: int = 256,  # 隐藏层维度
                 num_heads: int = 8,  # 多头注意力头数
                 num_feature_levels: int = 4):  # 特征金字塔层级数
        super().__init__()  # 调用父类nn.Module的初始化
        
        # 存储模型参数
        self.num_classes = num_classes  # 类别数
        self.num_queries = num_queries  # 查询数
        self.num_layers = num_layers  # Transformer层数
        self.hidden_dim = hidden_dim  # 特征维度
        
        # 1. 特征提取骨干网络(简化版ResNet+FPN)
        self.backbone = SimplifiedBackbone(hidden_dim, num_feature_levels)  # 初始化特征提取网络
        
        # 2. 可学习的对象查询嵌入
        self.query_embed = nn.Embedding(num_queries, hidden_dim)  # 形状为[num_queries, hidden_dim]
        
        # 3. Transformer解码器层
        self.transformer_layers = nn.ModuleList([  # 创建包含多个Transformer层的列表
            DETR3DLayer(hidden_dim, num_heads, num_feature_levels)
            for _ in range(num_layers)  # 创建num_layers个相同的层
        ])
        
        # 4. 预测头
        self.reference_points_head = nn.Linear(hidden_dim, 3)  # 预测3D参考点(x,y,z)
        self.bbox_head = nn.Linear(hidden_dim, 9)  # 预测3D边界框参数(中心点+尺寸+旋转)
        self.cls_head = nn.Linear(hidden_dim, num_classes + 1)  # 分类头(+1表示背景类)
        
        self._reset_parameters()  # 初始化模型参数
    
    def _reset_parameters(self):
        """初始化模型参数"""
        for p in self.parameters():  # 遍历所有可学习参数
            if p.dim() > 1:  # 只初始化维度大于1的参数(即矩阵)
                nn.init.xavier_uniform_(p)  # 使用Xavier均匀分布初始化
    
    def forward(self, 
                images: torch.Tensor,  # 输入图像张量[B, N_cams, 3, H, W]
                camera_matrices: torch.Tensor,  # 相机矩阵[B, N_cams, 3, 4]
                image_shapes: List[Tuple[int, int]]) -> dict:  # 各图像原始尺寸
        """
        前向传播过程:
        1. 提取多视角图像特征
        2. 初始化对象查询
        3. 通过Transformer层迭代优化查询
        4. 预测3D边界框和类别
        """
        batch_size, num_cams = images.shape[:2]  # 获取批次大小和相机数量
        
        # 1. 提取多维特征，
        multi_level_features = self.backbone(images)  # List of [B*N_cams, C, H_i, W_i]
        
        # 2. 初始化对象查询
        object_queries = self.query_embed.weight.unsqueeze(0).repeat(batch_size, 1, 1)  # [B, N_queries, C]
        
        # 3. 存储各层的预测结果
        all_predictions = []  # 用于存储每层的预测结果
        
        # 4. 通过Transformer层迭代优化
        for layer_idx, transformer_layer in enumerate(self.transformer_layers):
            # 4.1 预测当前查询对应的3D参考点(归一化坐标)
            reference_points = self.reference_points_head(object_queries).sigmoid()# [B, N_queries, 3]
            
            # 4.2 使用多视角特征更新查询
            object_queries = transformer_layer(
                object_queries,  # 当前查询
                multi_level_features,  # 多尺度特征
                reference_points,  # 3D参考点
                camera_matrices,  # 相机参数
                image_shapes,  # 图像尺寸
                batch_size,  # 批次大小
                num_cams  # 相机数量
            )
            
            # 4.3 使用更新后的查询进行预测
            bbox_pred = self.bbox_head(object_queries)  # [B, N_queries, 9]
            cls_pred = self.cls_head(object_queries)    # [B, N_queries, num_classes+1]
            
            # 4.4 存储当前层的预测结果
            all_predictions.append({
                'pred_boxes': bbox_pred,  # 边界框预测
                'pred_logits': cls_pred,  # 类别logits
                'reference_points': reference_points  # 参考点
            })
        
        # 返回所有预测结果和最终查询状态
        return {
            'predictions': all_predictions,  # 各层预测结果
            'final_queries': object_queries  # 最终查询状态
        }


class DETR3DLayer(nn.Module):
    """DETR3D的Transformer层
    主要功能:
    - 从多视角采样特征
    - 查询自注意力
    - 前馈网络
    """
    
    def __init__(self, hidden_dim: int, num_heads: int, num_feature_levels: int):  # 初始化函数
        super().__init__()  # 调用父类初始化
        
        # 存储层参数
        self.hidden_dim = hidden_dim  # 特征维度
        self.num_heads = num_heads  # 注意力头数
        self.num_feature_levels = num_feature_levels  # 特征层级数
        
        # 1. 多头自注意力机制
        self.self_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=0.1)
        
        # 2. 特征投影层
        self.feature_proj = nn.Linear(hidden_dim, hidden_dim)
        
        # 3. 归一化层和前馈网络
        self.norm1 = nn.LayerNorm(hidden_dim)  # 第一层归一化
        self.norm2 = nn.LayerNorm(hidden_dim)  # 第二层归一化
        
        # 4. 前馈网络(两层MLP)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),  # 扩展维度
            nn.ReLU(),  # 非线性激活
            nn.Dropout(0.1),  # 随机失活
            nn.Linear(hidden_dim * 4, hidden_dim),  # 降维
            nn.Dropout(0.1)  # 随机失活
        )
    
    def forward(self,
                queries: torch.Tensor,
                multi_level_features: List[torch.Tensor],
                reference_points: torch.Tensor,
                camera_matrices: torch.Tensor,
                image_shapes: List[Tuple[int, int]],
                batch_size: int,
                num_cams: int) -> torch.Tensor:
        """
        参数:
            queries: [B, N_queries, C] - 对象查询
            multi_level_features: [B*N_cams, C, H_i, W_i]列表 - 多层级特征
            reference_points: [B, N_queries, 3] - 3D参考点
            camera_matrices: [B, N_cams, 3, 4] - 相机矩阵
            image_shapes: (H, W)列表 - 图像尺寸
            batch_size: int - 批次大小
            num_cams: int - 相机数量
        """
        
        # 1. 从多视角和多层级采样特征
        sampled_features = self.sample_multi_view_features(
            reference_points, 
            multi_level_features, 
            camera_matrices,
            image_shapes,
            batch_size, 
            num_cams
        )  # [B, N_queries, C]
        
        # 2. 将采样特征添加到查询中
        queries = queries + sampled_features
        
        # 3. 对象查询间的自注意力
        queries_t = queries.transpose(0, 1)  # [N_queries, B, C]
        attn_queries, _ = self.self_attn(queries_t, queries_t, queries_t)
        attn_queries = attn_queries.transpose(0, 1)  # [B, N_queries, C]
        
        # 4. 残差连接和归一化
        queries = self.norm1(queries + attn_queries)
        
        # 5. 前馈网络
        ffn_output = self.ffn(queries)
        queries = self.norm2(queries + ffn_output)
        
        return queries  # 返回更新后的查询
    
    def sample_multi_view_features(self,
                                 reference_points: torch.Tensor,
                                 multi_level_features: List[torch.Tensor],
                                 camera_matrices: torch.Tensor,
                                 image_shapes: List[Tuple[int, int]],
                                 batch_size: int,
                                 num_cams: int) -> torch.Tensor:
        """
        从多相机视角和多特征层级采样特征
        
        参数:
            reference_points: [B, N_queries, 3] - 世界坐标系中的3D点
            multi_level_features: [B*N_cams, C, H_i, W_i]列表 - 多层级特征
            camera_matrices: [B, N_cams, 3, 4] - 相机变换矩阵
            image_shapes: (H, W)列表 - 图像尺寸
            batch_size: int - 批次大小
            num_cams: int - 相机数量
        
        返回:
            torch.Tensor: [B, N_queries, C] - 聚合后的特征
        """
        num_queries = reference_points.shape[1]  # 获取查询数量
        aggregated_features = []  # 存储聚合特征的列表
        
        for b in range(batch_size):  # 遍历每个批次
            batch_features = []  # 存储当前批次的特征
            valid_count = 0  # 有效相机计数
            
            for cam_idx in range(num_cams):  # 遍历每个相机
                # 获取当前批次和相机的变换矩阵
                cam_matrix = camera_matrices[b, cam_idx]  # [3, 4]
                
                # 将3D点转换为齐次坐标
                points_3d_homo = torch.cat([
                    reference_points[b], 
                    torch.ones(num_queries, 1, device=reference_points.device)
                ], dim=1)  # [N_queries, 4]
                
                # 投影到2D图像坐标
                points_2d_homo = torch.mm(points_3d_homo, cam_matrix.T)  # [N_queries, 3]
                points_2d = points_2d_homo[:, :2] / (points_2d_homo[:, 2:3] + 1e-8)  # [N_queries, 2]
                
                # 从所有层级采样当前相机的特征
                cam_features = []
                for level_idx, features in enumerate(multi_level_features):
                    H, W = features.shape[-2:]  # 获取特征图尺寸
                    
                    # 归一化坐标到[-1, 1]以便grid_sample使用
                    normalized_coords = points_2d.clone()
                    normalized_coords[:, 0] = 2.0 * points_2d[:, 0] / W - 1.0  # x坐标
                    normalized_coords[:, 1] = 2.0 * points_2d[:, 1] / H - 1.0  # y坐标
                    
                    # 检查点是否在图像边界内
                    valid_mask = (
                        (normalized_coords[:, 0] >= -1) & (normalized_coords[:, 0] <= 1) &
                        (normalized_coords[:, 1] >= -1) & (normalized_coords[:, 1] <= 1)
                    )
                    
                    # 获取当前相机和层级的特征图
                    feat_map = features[b * num_cams + cam_idx].unsqueeze(0)  # [1, C, H, W]
                    
                    # 使用双线性插值采样特征
                    sample_coords = normalized_coords.unsqueeze(0).unsqueeze(0)  # [1, 1, N_queries, 2]
                    sampled_feat = F.grid_sample(
                        feat_map, 
                        sample_coords, 
                        mode='bilinear', 
                        padding_mode='zeros',
                        align_corners=False
                    )  # [1, C, 1, N_queries]
                    
                    sampled_feat = sampled_feat.squeeze(0).squeeze(1).T  # [N_queries, C]
                    
                    # 应用有效掩码
                    sampled_feat[~valid_mask] = 0
                    
                    cam_features.append(sampled_feat)
                
                # 计算当前相机所有层级的平均特征
                if cam_features:
                    cam_feat_avg = torch.stack(cam_features, dim=0).mean(dim=0)  # [N_queries, C]
                    # 将当前相机的平均特征加入批次特征列表
                    batch_features.append(cam_feat_avg)
                    # 有效相机计数+1
                    valid_count += 1
            
            # 计算所有相机的平均特征
            if batch_features:
                # 对当前batch所有相机的特征求和后除以有效相机数(加1e-8防止除零)
                batch_feat_avg = torch.stack(batch_features, dim=0).sum(dim=0) / (valid_count + 1e-8)
            else:
                # 如果没有有效特征，则创建全零特征张量
                batch_feat_avg = torch.zeros(num_queries, self.hidden_dim, device=reference_points.device)
            
            # 将当前batch的平均特征加入聚合特征列表
            aggregated_features.append(batch_feat_avg)
        
        return torch.stack(aggregated_features, dim=0)  # [B, N_queries, C]


class SimplifiedBackbone(nn.Module):
    """简化版的backbone网络 (ResNet + FPN)"""
    
    def __init__(self, hidden_dim: int, num_levels: int = 4):
        super().__init__()
        
        self.hidden_dim = hidden_dim  # 隐藏层维度
        self.num_levels = num_levels  # 特征层级数
        
        # 简化的特征提取层
        self.conv_layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(3 if i == 0 else hidden_dim, hidden_dim, 3, 
                         stride=2**i if i > 0 else 1, padding=1),
                nn.BatchNorm2d(hidden_dim),  # 批归一化
                nn.ReLU(inplace=True),  # ReLU激活函数
                nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),  # 第二层卷积
                nn.BatchNorm2d(hidden_dim),  # 批归一化
                nn.ReLU(inplace=True)  # ReLU激活函数
            ) for i in range(num_levels)  # 为每个层级创建相同的结构
        ])
    
    def forward(self, images: torch.Tensor) -> List[torch.Tensor]:
        """
        Args:
            images: [B, N_cams, 3, H, W] 输入图像张量，形状为[批次,相机数,通道,高,宽]
        
        Returns:
            List of feature maps: [B*N_cams, C, H_i, W_i] 返回多尺度特征图列表
        """
        B, N, C, H, W = images.shape  # 获取输入形状
        
        # Reshape to process all images together
        # 重塑张量以便同时处理所有图像
        x = images.view(B * N, C, H, W)
        
        features = []  # 存储各层特征
        for i, layer in enumerate(self.conv_layers):
            x = layer(x)  # 通过当前层级
            features.append(x)  # 保存特征
            
        return features


def demo_forward_pass():
    """演示DETR3D的前向传播过程"""
    
    # 设置参数
    batch_size = 2  # 批次大小
    num_cams = 6  # 相机数量
    num_queries = 900  # 查询数量
    num_classes = 10  # 类别数
    image_size = (256, 256)  # 图像尺寸
    
    # 创建模型
    model = DETR3D(
        num_classes=num_classes,
        num_queries=num_queries,
        num_layers=6,
        hidden_dim=256
    )
    
    # 创建示例输入
    images = torch.randn(batch_size, num_cams, 3, *image_size)  # 随机图像
    camera_matrices = torch.randn(batch_size, num_cams, 3, 4)  # 相机变换矩阵(简化版)
    image_shapes = [image_size] * batch_size * num_cams  # 图像尺寸列表
    
    print("=" * 50)
    print("DETR3D Forward Pass Demo")
    print("=" * 50)
    
    print(f"Input shapes:")
    print(f"  Images: {images.shape}")  # 图像形状
    print(f"  Camera matrices: {camera_matrices.shape}")  # 相机矩阵形状
    print(f"  Number of queries: {num_queries}")  # 查询数量
    print(f"  Number of classes: {num_classes}")  # 类别数量
    
    # 前向传播
    model.eval()  # 评估模式
    with torch.no_grad():  # 不计算梯度
        outputs = model(images, camera_matrices, image_shapes)
    
    print(f"\nOutput structure:")
    print(f"  Number of layers: {len(outputs['predictions'])}")  # 输出层数
    
    # 显示每层的输出
    for i, pred in enumerate(outputs['predictions']):
        print(f"\n  Layer {i+1}:")
        print(f"    Bounding boxes: {pred['pred_boxes'].shape}")  # 边界框形状
        print(f"    Classification logits: {pred['pred_logits'].shape}")  # 分类logits
        print(f"    Reference points: {pred['reference_points'].shape}")  # 参考点
        
        # Show some statistics
        # 显示统计信息
        bbox_mean = pred['pred_boxes'].mean().item()  # 边界框均值
        cls_max_prob = torch.softmax(pred['pred_logits'], dim=-1).max().item()  # 最大类别概率
        
        print(f"    Bbox mean: {bbox_mean:.4f}")  # 边界框均值
        print(f"    Max class probability: {cls_max_prob:.4f}")  # 最大类别概率
    
    print(f"\nFinal queries shape: {outputs['final_queries'].shape}")  # 最终查询形状

    # 模拟后处理：获取置信度最高的检测结果
    final_predictions = outputs['predictions'][-1]  # Use last layer's predictions
    class_probs = torch.softmax(final_predictions['pred_logits'], dim=-1)  # 计算类别概率
    

    # 获取非背景类的最大概率
    object_probs = class_probs[:, :, :-1].max(dim=-1)[0]  # [B, N_queries]
    
    print(f"\nPost-processing example:")
    for b in range(batch_size):

        # 获取置信度大于阈值的检测
        confident_mask = object_probs[b] > 0.5  # 置信度掩码
        num_detections = confident_mask.sum().item()  # 检测数量
        
        print(f"  Batch {b+1}: {num_detections} confident detections (>0.5 confidence)")
        
        if num_detections > 0:
            confident_boxes = final_predictions['pred_boxes'][b][confident_mask]  # 置信框
            confident_probs = object_probs[b][confident_mask]  # 置信概率
            
            print(f"    Top detection - Confidence: {confident_probs.max().item():.4f}")  # 最高置信度
            print(f"    Box params: {confident_boxes[confident_probs.argmax()].tolist()}")  # 框参数


if __name__ == "__main__":
    demo_forward_pass()

DETR3D Forward Pass Demo
Input shapes:
  Images: torch.Size([2, 6, 3, 256, 256])
  Camera matrices: torch.Size([2, 6, 3, 4])
  Number of queries: 900
  Number of classes: 10

Output structure:
  Number of layers: 6

  Layer 1:
    Bounding boxes: torch.Size([2, 900, 9])
    Classification logits: torch.Size([2, 900, 11])
    Reference points: torch.Size([2, 900, 3])
    Bbox mean: 0.0019
    Max class probability: 0.8814

  Layer 2:
    Bounding boxes: torch.Size([2, 900, 9])
    Classification logits: torch.Size([2, 900, 11])
    Reference points: torch.Size([2, 900, 3])
    Bbox mean: 0.0449
    Max class probability: 0.8740

  Layer 3:
    Bounding boxes: torch.Size([2, 900, 9])
    Classification logits: torch.Size([2, 900, 11])
    Reference points: torch.Size([2, 900, 3])
    Bbox mean: 0.3228
    Max class probability: 0.9158

  Layer 4:
    Bounding boxes: torch.Size([2, 900, 9])
    Classification logits: torch.Size([2, 900, 11])
    Reference points: torch.Size([2, 900, 3])
 